In [20]:
import torch
from torch import nn
from d2l import torch as d2l

In [21]:
# Input–Output Pair

X = torch.ones((6, 8))
X[:, 2:6] = 0

target_kernel = torch.tensor([
    [1.0, -1.0]
])

Y = d2l.corr2d(
    X,
    target_kernel,
)

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print(Y)

X shape: torch.Size([6, 8])
Y shape: torch.Size([6, 7])
tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])


In [22]:
# Convolution Layer & 4D Tensor

conv2d = nn.LazyConv2d(
    out_channels=1,
    kernel_size=(1, 2),
    bias=False,
)

X_train = X.reshape((1, 1, 6, 8))
Y_train = Y.reshape((1, 1, 6, 7))

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)

X_train shape: torch.Size([1, 1, 6, 8])
Y_train shape: torch.Size([1, 1, 6, 7])


In [23]:
# Kernel(Convolution Window) Learning

learning_rate = 3e-2

for epoch in range(20):
    Y_hat = conv2d(X_train)
    loss = (Y_hat - Y_train).pow(2)
    loss_sum = loss.sum()
    
    conv2d.zero_grad()
    loss_sum.backward()
    
    gradient = conv2d.weight.grad
    
    if gradient is None:
        raise RuntimeError(
            "Kernel gradient was not computed"
        )
    
    with torch.no_grad():
        conv2d.weight.add_(
            gradient,
            alpha=-learning_rate,
        )
        
    print(
        f"epoch {epoch + 1}, "
        f"loss {loss.sum().item():.3f}"
    )
    
learned_kernel = (
    conv2d.weight
    .detach()
    .reshape(1, 2)
)

print(
    "\ntarget kernel:",
    target_kernel,
)
print(
    "learned kernel:",
    learned_kernel,
)

epoch 1, loss 24.766
epoch 2, loss 13.384
epoch 3, loss 7.555
epoch 4, loss 4.421
epoch 5, loss 2.660
epoch 6, loss 1.633
epoch 7, loss 1.017
epoch 8, loss 0.639
epoch 9, loss 0.404
epoch 10, loss 0.257
epoch 11, loss 0.164
epoch 12, loss 0.104
epoch 13, loss 0.067
epoch 14, loss 0.043
epoch 15, loss 0.027
epoch 16, loss 0.017
epoch 17, loss 0.011
epoch 18, loss 0.007
epoch 19, loss 0.005
epoch 20, loss 0.003

target kernel: tensor([[ 1., -1.]])
learned kernel: tensor([[ 0.9943, -1.0055]])
